# Ming-3.0-tts-0.5b Cookbook

One runnable example per capability. Every cell calls the same entry point,
`MingTTS.synthesize`, and differs only in the `task` and its controls.

Capabilities new in 3.0 are marked **new**.

In [ ]:
# Run from the repository root so the local modules resolve.
%cd /path/to/Ming-3.0-tts

In [ ]:
from IPython.display import Audio, display

from ming_tts import MingTTS

MODEL_PATH = "inclusionAI/Ming-3.0-tts-0.5b"  # or a local checkpoint directory
model = MingTTS(MODEL_PATH, device="cuda:0")

# A reference recording and its transcript, used by the cloning examples.
PROMPT_WAV = "data/voice_presets/voice_audio/lingguang.wav"
PROMPT_TEXT = "诶，你今天有空吗？我们能不能聊聊这件事啊？"


def speak(**kwargs):
    waveform = model.synthesize(**kwargs)
    display(Audio(waveform.float().numpy(), rate=model.sample_rate))
    return waveform

## Zero-shot voice cloning

Give one reference recording and its transcript. The transcript matters: the
model conditions on the pair, so a wrong transcript degrades similarity.

In [ ]:
speak(
    task="TTS",
    text="雨后的山谷很安静，远处亮起了一盏温暖的灯。",
    prompt_wav_path=PROMPT_WAV,
    prompt_text=PROMPT_TEXT,
    use_spk_emb=True,
)

## Paralinguistic control **new**

Tags go inline, anywhere in the sentence, in Chinese or English. They are
routed around text normalization, so they survive intact.

Available tags: `[Breathing]` `[Cough]` `[Sigh]` `[Laughter]` `[Surprise-ah]`
`[Surprise-oh]` `[Dissatisfaction-hnn]` `[Uhm]` `[Confirmation-en]`
`[Question-ei]` `[Question-ah]` `[Question-en]` `[Question-oh]` `[Question-huh]`

In [ ]:
speak(
    task="TTS",
    text="[Laughter] 你居然真的把那封信藏在旧钟后面了。",
    prompt_wav_path=PROMPT_WAV,
    prompt_text=PROMPT_TEXT,
    use_spk_emb=True,
)

In [ ]:
# Tags compose, and work mid-sentence in English too.
speak(
    task="TTS",
    text="I checked the drawer twice [Sigh] and it was empty [Uhm] again.",
    language_id="en",
    prompt_wav_path=PROMPT_WAV,
    prompt_text=PROMPT_TEXT,
    use_spk_emb=True,
)

## Multilingual **new**

German, Spanish, French, Korean and Japanese join Chinese and English.
`language_id` is required here — it both adds the target-language line to the
prompt and keeps the text away from the Chinese/English normalizer, which
would otherwise strip Japanese kana.

The voice carries across languages, so a Chinese reference can speak Japanese.

In [ ]:
for language, sentence in [
    ("ja", "あなた、ご自分のことばかりですのね。"),
    ("ko", "오늘 저녁에는 비가 올 것 같아요."),
    ("de", "Der Garten bleibt bis Sonnenuntergang geöffnet."),
    ("fr", "Le vieux phare veille encore sur la baie."),
    ("es", "El jardín permanece abierto hasta el atardecer."),
]:
    print(language, sentence)
    speak(
        task="MULTILINGUAL",
        text=sentence,
        language_id=language,
        prompt_wav_path=PROMPT_WAV,
        prompt_text=PROMPT_TEXT,
        use_spk_emb=True,
    )

## Free-form instruction **new**

Describe the delivery in one sentence. Reference-free generation is recommended for both models; adding a reference is not recommended. For 0.5B, `use_zero_spk_emb=True` supplies a zeroed speaker
embedding and lets the instruction define the voice.

In [ ]:
speak(
    task="FREEFORM",
    text="The old radio crackled softly as the storm approached.",
    language_id="en",
    instruction="Speak in a warm, intimate storytelling voice, with a gentle "
                "pace and a hint of nostalgia.",
    use_zero_spk_emb=True,
)

## Voice design

The same idea expressed as a voice description rather than a delivery
instruction. Text is not split into fragments here, because the style applies
to the utterance as a whole.

In [ ]:
speak(
    task="VOICE_DESIGN",
    text="冰川在漫长岁月中缓缓移动，留下深刻的痕迹。",
    controls={"style": "成熟沉稳的男性纪录片旁白，吐字清晰，节奏舒缓。"},
    use_zero_spk_emb=True,
)

## Basic attribute control

`speed` is `慢速 / 中速 / 快速`; `pitch` and `volume` are `低 / 中 / 高`.
The three move independently, so changing one does not drag the others.

In [ ]:
for speed in ["慢速", "中速", "快速"]:
    print("speed:", speed)
    speak(
        task="BASIC",
        text="今晚的观测将在云层散开后开始。",
        controls={"speed": speed, "pitch": "中", "volume": "中"},
        prompt_wav_path=PROMPT_WAV,
        prompt_text=PROMPT_TEXT,
        use_spk_emb=True,
    )

## Emotion

`高兴 / 悲伤 / 愤怒 / 恐惧 / 厌恶 / 惊讶 / 中性`.

In [ ]:
for emotion in ["高兴", "悲伤", "愤怒"]:
    print("emotion:", emotion)
    speak(
        task="EMOTION",
        text="所有的测试都通过了。",
        controls={"emotion": emotion},
        prompt_wav_path="data/wavs/emotion_prompt.wav",
        use_spk_emb=True,
    )

## Dialect

`cantonese` or `chuanyu dialect`. This task takes no `language_id`: a dialect
is selected through the control, not the target language. Passing one anyway
raises a warning and is ignored.

In [ ]:
speak(
    task="DIALECT",
    text="今日天气几好，我哋一齐去海边行下啦。",
    controls={"dialect": "cantonese"},
    prompt_wav_path="data/wavs/yue_prompt.wav",
    use_spk_emb=True,
)

In [ ]:
speak(
    task="DIALECT",
    text="今天天气巴适得很，我们一起出去转哈嘛。",
    controls={"dialect": "chuanyu dialect"},
    prompt_wav_path=PROMPT_WAV,
    prompt_text=PROMPT_TEXT,
    use_spk_emb=True,
)

## Built-in premium voices

No reference recording: five English voices and 479 Chinese voices.
Recommended: **US_James**, **US_Stella**, **灵小柔**, **灵楷**.
See the [full voice list](instructions.md#built-in-premium-voices).

In [ ]:
for voice in ["US_James", "US_Stella", "US_Emma", "UK_George", "UK_Charlotte"]:
    print("voice:", voice)
    speak(
        task="IP",
        text="The garden will remain open until sunset, "
             "provided the weather stays clear.",
        language_id="en",
        controls={"ip": voice},
        use_zero_spk_emb=True,
    )

## Text normalization

Numbers, dates, units and formulas are expanded before synthesis. This needs
`pynini`; it is loaded lazily, so construct the model with
`text_normalization=False` to skip it entirely.

In [ ]:
speak(
    task="TTS",
    text="订单号 20250317，共 3 件商品，合计 ¥1,280.50，预计 3-5 天送达。",
    prompt_wav_path=PROMPT_WAV,
    prompt_text=PROMPT_TEXT,
    use_spk_emb=True,
)

In [ ]:
# Compare against the un-normalized reading.
plain = MingTTS(MODEL_PATH, device="cuda:0", text_normalization=False)
waveform = plain.synthesize(
    task="TTS",
    text="订单号 20250317，共 3 件商品，合计 ¥1,280.50，预计 3-5 天送达。",
    prompt_wav_path=PROMPT_WAV,
    prompt_text=PROMPT_TEXT,
    use_spk_emb=True,
)
display(Audio(waveform.float().numpy(), rate=plain.sample_rate))